<a href="https://colab.research.google.com/github/Dill77/Birdcall_Individual_Clips_Audiocondenser/blob/main/Birdcall_Detector_Individual_Clips_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#This program can simply run all at the start, get prompted with an audio file
#to upload, and then select desired audio file. Have BirdNET run and detect
#all species present, and splice out the audio into separate clips
#I highly recommend scanning any files through Chirpity first, as this is using the same
#background process via BirdNET, but takes significantly longer to analyse and cut all clips down

In [1]:
#Run this section the first time you want to load the program, and make sure to restart the
#Colab session before you try and run the main block!
#If you get an error message, this little piece is likely the bug fix for you
print('--- Applying fixes ---')
!pip install --upgrade numba resampy
print("Upgraded numba and resampy.")

--- Applying fixes ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 17.9 MB/s eta 0:00:00
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: numba
    Found existing installation: numba 0.60.0
    Uninstalling numba-0.60.0:
      Successfully uninstalled numba-0.60.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
Upgraded numba and resampy.


In [1]:
# Bird Call Clip Extractor (application using BirdNET)

#This notebook takes an uploaded audio file, uses **BirdNET** (via the `birdnetlib` Python package) to detect bird vocalizations, and saves **each detected call as its own audio file**, named with the timestamp (from the original recording) at which it occurred.

#**How it works**
#1. Program will install all dependencies(BirdNET model + audio tools), so that (hopefully!) no other downloading needs to take place!
#2. Upload your audio file.
#3. Run BirdNET to detect bird calls with timestamps and confidence scores (level can be manually changed pre-upload!)
#4. Merge nearby detections and add a small padding so calls aren't cut short or repeated if calls have a small time lag in them
#5. Export each merged segment as its own file, named with its start/end timestamp
#6. Download all clips as a single zip.
# Quick note: using 'birdnetlib', which is just the python version of the same thing Chirpity uses - to get around access issues (and the fact that Chirpity doesn't have the function of cutting audio files!)

## 1. Install dependencies
#This takes a minute or two the first time you run it, but afterwards is faster
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q birdnetlib tensorflow pydub
print("Done installing dependencies.")

## 2. Upload your audio file
#Supports common formats (.wav in the case of our Mpala data)
from google.colab import files
import os

uploaded = files.upload()
assert len(uploaded) > 0, "No file uploaded."
INPUT_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {INPUT_PATH} ({os.path.getsize(INPUT_PATH)/1e6:.2f} MB)")

## 3. Configurable settings before running!

#- `MIN_CONFIDENCE`: is the minimum BirdNET confidence (0-1) for a detection to count as a real bird call. I think around 0.3 is reasonable, but can edit to be more or less tolerant!
#- `PADDING_SECONDS`: extra audio kept before/after each detected call so it isn't clipped abruptly. Change if audio has too much or too littel silence before and after (might be helpful to increase for ML training in the future)
#- `MERGE_GAP_SECONDS`: if two detections (after padding) are closer together than this, they're merged into one clip instead of being split apart for easier recognition by future models
#- `RECORDING_START`: Optional! As all files were recorded starting at a specific real-world date/time, user can set this (e.g. `datetime(2024, 5, 10, 6, 30, 0)`) and clip filenames will use corresponding real clock times. If left as `None`, filenames use offsets from the start of the file instead (e.g. `00-01-23_to_00-01-27`).
#- `LAT` / `LON` / `DATE`: optional. Giving BirdNET a location and date obviously improves detection accuracy. Leave as `None` to skip. Default should be the coordinates of Mpala Ranch - but currently bugged when changing location! DO NOT EDIT!!!

from datetime import datetime, timedelta

MIN_CONFIDENCE = 0.3
PADDING_SECONDS = 0.5
MERGE_GAP_SECONDS = 1.0

# Optional: real-world start time of the recording, for clock-time filenames.
# e.g. RECORDING_START = datetime(2024, 5, 10, 6, 30, 0)
RECORDING_START = None

# Optional, improves detection accuracy. Leave as None to skip. Currently bugged DO NOT EDIT!
LAT = None
LON = None
DATE = None     # e.g. datetime(2024, 5, 10)

## 4. Run BirdNET detection
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer

print("Loading BirdNET model (first run downloads model weights)...")
analyzer = Analyzer()

recording_kwargs = dict(min_conf=MIN_CONFIDENCE)
if LAT is not None and LON is not None:
    recording_kwargs["lat"] = LAT
    recording_kwargs["lon"] = LON
if DATE is not None:
    recording_kwargs["date"] = DATE

recording = Recording(analyzer, INPUT_PATH, **recording_kwargs)
print("Analyzing audio for bird calls...")
recording.analyze()

detections = recording.detections
print(f"Found {len(detections)} raw detections above confidence {MIN_CONFIDENCE}.")

for d in sorted(detections, key=lambda x: x['start_time'])[:10]:
    print(f"  {d['start_time']:.1f}s - {d['end_time']:.1f}s  {d['common_name']}  ({d['confidence']:.2f})")
if len(detections) > 10:
    print(f"  ... and {len(detections) - 10} more")

## 5. Merge detections into clip segments
def build_segments(detections, padding, merge_gap):
    if not detections:
        return []

    intervals = sorted(
        (max(0.0, d['start_time'] - padding), d['end_time'] + padding)
        for d in detections
    )

    merged = [intervals[0]]
    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end + merge_gap:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged

segments = build_segments(detections, PADDING_SECONDS, MERGE_GAP_SECONDS)

print(f"{len(segments)} clip(s) will be produced.")
for start, end in segments:
    print(f"  clip {start:.1f}s - {end:.1f}s  ({end - start:.1f}s)")

## 6. Export each segment as its own file

#Filenames encode the timestamp of the clip within the original recording:
#- If `RECORDING_START` was set, filenames use real clock time, e.g. `birdcall_2024-05-10_06-31-23_to_06-31-27.wav`.
#- Otherwise, filenames use elapsed offset from the start of the file, e.g. `birdcall_00-01-23_to_00-01-27.wav` (hh-mm-ss).

from pydub import AudioSegment
import os, shutil

assert segments, "No bird calls detected above the confidence threshold -- nothing to extract. Try lowering MIN_CONFIDENCE."

audio = AudioSegment.from_file(INPUT_PATH)
duration_s = len(audio) / 1000.0
print(f"Original duration: {duration_s:.1f}s")

def offset_str(seconds):
    td = timedelta(seconds=int(seconds))
    total = int(td.total_seconds())
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}-{m:02d}-{s:02d}"

def clock_str(dt):
    return dt.strftime("%Y-%m-%d_%H-%M-%S")

OUTPUT_DIR = "bird_clips"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

clip_paths = []
for i, (start, end) in enumerate(segments, start=1):
    start_ms = int(max(0, start) * 1000)
    end_ms = int(min(duration_s, end) * 1000)
    clip = audio[start_ms:end_ms]

    if RECORDING_START is not None:
        clip_start_dt = RECORDING_START + timedelta(seconds=start)
        clip_end_dt = RECORDING_START + timedelta(seconds=end)
        tag = f"{clock_str(clip_start_dt)}_to_{clip_end_dt.strftime('%H-%M-%S')}"
    else:
        tag = f"{offset_str(start)}_to_{offset_str(end)}"

    filename = f"birdcall_{i:03d}_{tag}.wav"
    filepath = os.path.join(OUTPUT_DIR, filename)
    clip.export(filepath, format="wav")
    clip_paths.append(filepath)
    print(f"Saved: {filepath}  ({(end - start):.1f}s)")

print(f"\nExported {len(clip_paths)} clip(s) to '{OUTPUT_DIR}/'.")

## 7. Download all clips as a zip
import shutil
from google.colab import files as colab_files

zip_base = "bird_clips"
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"Zipped: {zip_path}")
colab_files.download(zip_path)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires watchdog<7,>=6, but you have watchdog 2.1.9 which is incompatible.
Done installing dependencies.


Saving file_1682114400.wav to file_1682114400.wav
Uploaded: file_1682114400.wav (460.80 MB)


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Loading BirdNET model (first run downloads model weights)...
Labels loaded.
load model True


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.
Analyzing audio for bird calls...
read_audio_data
read_audio_data: complete, read  1200 chunks.
analyze_recording file_1682114400.wav
Found 0 raw detections above confidence 0.3.
0 clip(s) will be produced.


AssertionError: No bird calls detected above the confidence threshold -- nothing to extract. Try lowering MIN_CONFIDENCE.

In [2]:
# Ensure Analyzer is initialized (it should be from previous cell execution, but included for robustness!)
from birdnetlib.analyzer import Analyzer
from pydub import AudioSegment
import os, shutil
from datetime import datetime, timedelta # Ensure datetime and timedelta are imported here

# Function definitions from original notebook, moved here for chunking context
def offset_str(seconds):
    # timedelta is now imported at the top of the cell
    td = timedelta(seconds=int(seconds))
    total = int(td.total_seconds())
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}-{m:02d}-{s:02d}"

def build_segments(detections, padding, merge_gap):
    if not detections:
        return []

    intervals = sorted(
        (max(0.0, d['start_time'] - padding), d['end_time'] + padding)
        for d in detections
    )

    merged = [intervals[0]]
    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end + merge_gap:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged


if 'analyzer' not in locals(): # Check if analyzer is not already defined
    print("Initializing BirdNET analyzer...")
    analyzer = Analyzer()

# Get full audio duration
full_audio = AudioSegment.from_file(INPUT_PATH)
full_duration_seconds = len(full_audio) / 1000.0
print(f"Total audio duration: {full_duration_seconds:.1f}s")

CHUNK_LENGTH_SECONDS = 10 * 60  # 10 minutes per chunk
all_detections_from_chunks = []
processed_clip_paths = [] # List to store paths of all processed clips

# Prepare output directory
OUTPUT_DIR = "bird_clips"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

print(f"\nProcessing audio in {CHUNK_LENGTH_SECONDS/60:.0f}-minute chunks...")

for i, start_s in enumerate(range(0, int(full_duration_seconds), int(CHUNK_LENGTH_SECONDS))):
    end_s = min(start_s + CHUNK_LENGTH_SECONDS, full_duration_seconds)
    print(f"\n--- Processing chunk {i+1}: {offset_str(start_s)} to {offset_str(end_s)} ---")

    chunk_audio = full_audio[start_s * 1000 : end_s * 1000]
    chunk_filepath = os.path.join(OUTPUT_DIR, f"temp_chunk_{i:03d}.wav")
    chunk_audio.export(chunk_filepath, format="wav")

    # Run BirdNET on the chunk
    chunk_recording = Recording(analyzer, chunk_filepath, **recording_kwargs)
    print(f"Analyzing chunk {i+1}...")
    chunk_recording.analyze()

    # Adjust detection timestamps to original file's time and collect
    for d in chunk_recording.detections:
        d['start_time'] += start_s
        d['end_time'] += start_s
        all_detections_from_chunks.append(d)

    # Clean up temporary chunk file
    os.remove(chunk_filepath)

print(f"\nFound {len(all_detections_from_chunks)} raw detections across all chunks above confidence {MIN_CONFIDENCE}.")

# Assign to 'detections' variable for compatibility with subsequent merging logic
detections = all_detections_from_chunks

Total audio duration: 3600.0s

Processing audio in 10-minute chunks...

--- Processing chunk 1: 00-00-00 to 00-10-00 ---
Analyzing chunk 1...
read_audio_data
read_audio_data: complete, read  200 chunks.
analyze_recording temp_chunk_000.wav

--- Processing chunk 2: 00-10-00 to 00-20-00 ---
Analyzing chunk 2...
read_audio_data
read_audio_data: complete, read  200 chunks.
analyze_recording temp_chunk_001.wav

--- Processing chunk 3: 00-20-00 to 00-30-00 ---
Analyzing chunk 3...
read_audio_data
read_audio_data: complete, read  200 chunks.
analyze_recording temp_chunk_002.wav

--- Processing chunk 4: 00-30-00 to 00-40-00 ---
Analyzing chunk 4...
read_audio_data
read_audio_data: complete, read  200 chunks.
analyze_recording temp_chunk_003.wav

--- Processing chunk 5: 00-40-00 to 00-50-00 ---
Analyzing chunk 5...
read_audio_data
read_audio_data: complete, read  200 chunks.
analyze_recording temp_chunk_004.wav

--- Processing chunk 6: 00-50-00 to 01-00-00 ---
Analyzing chunk 6...
read_audio_da

In [3]:
from datetime import datetime, timedelta # Ensure datetime and timedelta are imported here

print(f"\nMerging {len(detections)} detections and exporting clips...")

# Merge detections from all chunks into final segments
sesgments = build_segments(detections, PADDING_SECONDS, MERGE_GAP_SECONDS)

print(f"{len(segments)} final clip(s) will be produced.")

assert segments, "No bird calls detected above the confidence threshold after chunking -- nothing to extract. Try lowering MIN_CONFIDENCE."

# Adding clock_str definition from original notebook for this cell's context
def clock_str(dt):
    # datetime is now imported at the top of the cell
    return dt.strftime("%Y-%m-%d_%H-%M-%S")

for i, (start, end) in enumerate(segments, start=1):
    start_ms = int(max(0, start) * 1000)
    end_ms = int(min(full_duration_seconds, end) * 1000) # Use full_duration_seconds
    clip = full_audio[start_ms:end_ms]

    if RECORDING_START is not None:
        # timedelta is now imported at the top of the cell
        clip_start_dt = RECORDING_START + timedelta(seconds=start)
        clip_end_dt = RECORDING_START + timedelta(seconds=end)
        tag = f"{clock_str(clip_start_dt)}_to_{clip_end_dt.strftime('%H-%M-%S')}"
    else:
        tag = f"{offset_str(start)}_to_{offset_str(end)}"

    filename = f"birdcall_{i:03d}_{tag}.wav"
    filepath = os.path.join(OUTPUT_DIR, filename)
    clip.export(filepath, format="wav")
    processed_clip_paths.append(filepath) # Use the new list
    print(f"Saved: {filepath}  ({(end - start):.1f}s)")

print(f"\nExported {len(processed_clip_paths)} clip(s) to '{OUTPUT_DIR}/'.")


Merging 0 detections and exporting clips...
0 final clip(s) will be produced.


AssertionError: No bird calls detected above the confidence threshold after chunking -- nothing to extract. Try lowering MIN_CONFIDENCE.

In [4]:
#Run this bottom block if you would also like randomised background sample audio
#This can be useful to spot-check the detector, or to obtain background audio for
#the future Perchv2 re-training!

In [6]:
## 8. Build background (non-bird) clip intervals from the gaps between bird segments

# Run this AFTER the cell that produces `segments` (the merged bird-call intervals)
# and after `full_audio`/`full_duration_seconds` (or `audio`/`duration_s` in the
# non-chunked version) exist.

import random
import shutil
from google.colab import files as colab_files

BG_CLIP_MIN_SECONDS = 5.0   # shortest allowed background clip
BG_CLIP_MAX_SECONDS = 10.0  # longest allowed background clip

# Use whichever audio/duration variables exist in this session
source_audio = full_audio if 'full_audio' in globals() else audio
total_duration = full_duration_seconds if 'full_duration_seconds' in globals() else duration_s

def build_gap_segments(bird_segments, total_duration, min_len=BG_CLIP_MIN_SECONDS):
    """Complement of the bird-call segments: stretches of audio with no bird call,
    each at least min_len seconds long (shorter gaps are skipped, not padded)."""
    gaps = []
    cursor = 0.0
    for start, end in sorted(bird_segments):
        if start - cursor >= min_len:
            gaps.append((cursor, start))
        cursor = max(cursor, end)
    if total_duration - cursor >= min_len:
        gaps.append((cursor, total_duration))
    return gaps

def chunk_gap_into_clips(gap_start, gap_end, min_len=BG_CLIP_MIN_SECONDS, max_len=BG_CLIP_MAX_SECONDS):
    """Slice one gap into back-to-back clips of random length between min_len and
    max_len seconds. Any leftover shorter than min_len is merged into the last clip
    rather than dropped."""
    clips = []
    cursor = gap_start
    while gap_end - cursor >= min_len:
        remaining = gap_end - cursor
        length = random.uniform(min_len, min(max_len, remaining))
        if remaining - length < min_len:
            length = remaining  # absorb the small leftover instead of discarding it
        clip_end = cursor + length
        clips.append((cursor, clip_end))
        cursor = clip_end
    return clips

gap_segments = build_gap_segments(segments, total_duration)
print(f"{len(gap_segments)} background gap(s) found (>= {BG_CLIP_MIN_SECONDS:.0f}s, no bird calls).")

background_clip_intervals = []
for g_start, g_end in gap_segments:
    background_clip_intervals.extend(chunk_gap_into_clips(g_start, g_end))

print(f"{len(background_clip_intervals)} background clip(s) will be produced.")
for start, end in background_clip_intervals:
    print(f"  bg clip {start:.1f}s - {end:.1f}s  ({end - start:.1f}s)")


## 9. Export background clips and zip them

BG_OUTPUT_DIR = "background_clips"
if os.path.exists(BG_OUTPUT_DIR):
    shutil.rmtree(BG_OUTPUT_DIR)
os.makedirs(BG_OUTPUT_DIR)

assert background_clip_intervals, "No background gaps long enough to clip -- try lowering BG_CLIP_MIN_SECONDS."

bg_clip_paths = []
for i, (start, end) in enumerate(background_clip_intervals, start=1):
    start_ms = int(max(0, start) * 1000)
    end_ms = int(min(total_duration, end) * 1000)
    clip = source_audio[start_ms:end_ms]

    if RECORDING_START is not None:
        clip_start_dt = RECORDING_START + timedelta(seconds=start)
        clip_end_dt = RECORDING_START + timedelta(seconds=end)
        tag = f"{clock_str(clip_start_dt)}_to_{clip_end_dt.strftime('%H-%M-%S')}"
    else:
        tag = f"{offset_str(start)}_to_{offset_str(end)}"

    filename = f"background_{i:03d}_{tag}.wav"
    filepath = os.path.join(BG_OUTPUT_DIR, filename)
    clip.export(filepath, format="wav")
    bg_clip_paths.append(filepath)
    print(f"Saved: {filepath}  ({(end - start):.1f}s)")

print(f"\nExported {len(bg_clip_paths)} background clip(s) to '{BG_OUTPUT_DIR}/'.")

bg_zip_base = "background_clips"
bg_zip_path = shutil.make_archive(bg_zip_base, "zip", BG_OUTPUT_DIR)
print(f"Zipped: {bg_zip_path}")
colab_files.download(bg_zip_path)

1 background gap(s) found (>= 5s, no bird calls).
485 background clip(s) will be produced.
  bg clip 0.0s - 7.2s  (7.2s)
  bg clip 7.2s - 13.2s  (5.9s)
  bg clip 13.2s - 22.9s  (9.7s)
  bg clip 22.9s - 30.6s  (7.7s)
  bg clip 30.6s - 38.1s  (7.4s)
  bg clip 38.1s - 47.3s  (9.2s)
  bg clip 47.3s - 56.4s  (9.1s)
  bg clip 56.4s - 64.2s  (7.8s)
  bg clip 64.2s - 72.7s  (8.5s)
  bg clip 72.7s - 79.8s  (7.1s)
  bg clip 79.8s - 87.7s  (7.9s)
  bg clip 87.7s - 96.1s  (8.4s)
  bg clip 96.1s - 104.2s  (8.1s)
  bg clip 104.2s - 112.3s  (8.1s)
  bg clip 112.3s - 118.1s  (5.8s)
  bg clip 118.1s - 127.7s  (9.6s)
  bg clip 127.7s - 133.7s  (6.0s)
  bg clip 133.7s - 142.7s  (9.0s)
  bg clip 142.7s - 151.8s  (9.0s)
  bg clip 151.8s - 158.8s  (7.1s)
  bg clip 158.8s - 167.8s  (8.9s)
  bg clip 167.8s - 173.6s  (5.8s)
  bg clip 173.6s - 183.0s  (9.4s)
  bg clip 183.0s - 191.2s  (8.2s)
  bg clip 191.2s - 196.4s  (5.2s)
  bg clip 196.4s - 205.8s  (9.4s)
  bg clip 205.8s - 213.8s  (8.0s)
  bg clip 213.8s - 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>